# Análise exploratória de cancelamentos

[ Abrir no Colab ](https://colab.research.google.com/github/danielfernandomartins/analise-dados-clientes/blob/main/analise_cancelamentos.ipynb)

Estudo de portfólio. O CSV original não está versionado; forneça `cancelamentos.csv` para executar. Origem e licença ainda precisam ser documentadas.

## Registro da execução anterior

As saídas salvas na versão anterior registravam 50.000 linhas, 49.996 após remoção de nulos e 28.393 cancelamentos (56,79%). Contratos mensais: 9.884 registros, todos com cancelamento. No recorte sem contratos mensais: 18.509 / 40.112 (46,14%). Com filtros adicionais de menos de 5 ligações e até 20 dias de atraso: 4.821 / 26.267 (18,35%).

Esses números são históricos, não uma nova execução. As diferenças entre taxas refletem seleção de amostras, não efeito de uma ação de retenção. O padrão de 100% nos contratos mensais exige investigar a construção da base.


In [ ]:
# Instalação das bibliotecas necessárias (caso não estejam instaladas)
!pip install pandas plotly nbformat -q

# Importação das bibliotecas
import pandas as pd
import plotly.express as px

print("Bibliotecas importadas com sucesso!")

In [ ]:
from pathlib import Path

arquivo = Path("cancelamentos.csv")
if not arquivo.is_file():
    raise FileNotFoundError("Forneça o CSV original cancelamentos.csv na pasta do notebook. A base não está incluída no repositório.")

tabela = pd.read_csv(arquivo)
colunas_necessarias = {"cancelou", "duracao_contrato", "assinatura", "ligacoes_callcenter", "dias_atraso"}
faltantes = colunas_necessarias - set(tabela.columns)
if faltantes:
    raise ValueError(f"Colunas ausentes: {sorted(faltantes)}")
if not tabela["cancelou"].dropna().isin([0, 1]).all():
    raise ValueError("cancelou deve conter apenas 0, 1 ou nulo.")
tabela = tabela.drop(columns=["CustomerID"], errors="ignore")
display(tabela.head())


In [ ]:
linhas_originais = len(tabela)
display(tabela.isna().sum().to_frame("nulos"))
tabela = tabela.dropna().copy()
if tabela.empty:
    raise ValueError("A base ficou vazia após a remoção de nulos.")
print(f"Linhas originais: {linhas_originais}; válidas: {len(tabela)}; removidas: {linhas_originais - len(tabela)}")
# Avaliar o padrão dos dados ausentes antes de aplicar esta estratégia a outra base.
tabela.info()


In [ ]:
# Quantidade de pessoas que cancelaram (1.0) e que não cancelaram (0.0)
display(tabela["cancelou"].value_counts())

# Proporção em formato percentual
display(tabela["cancelou"].value_counts(normalize=True).map("{:.1%}".format))

In [ ]:
resumo_contratos = tabela.groupby("duracao_contrato")["cancelou"].agg(clientes="size", cancelamentos="sum", taxa="mean")
resumo_contratos["taxa_pct"] = resumo_contratos["taxa"] * 100
display(resumo_contratos)


In [ ]:
# Recorte descritivo: preserva a base completa para as demais análises.
sem_mensais = tabela.loc[tabela["duracao_contrato"] != "Monthly"].copy()
print(f"Recorte sem contratos mensais: {len(sem_mensais)} clientes")
print(f"Taxa no recorte: {sem_mensais['cancelou'].mean():.2%}")
# Não representa redução causada por uma intervenção.


In [ ]:
# Comparação na base completa, sem excluir contratos mensais.
display(tabela.groupby("assinatura")["cancelou"].agg(clientes="size", taxa="mean"))


In [ ]:
# Distribuições na base completa. Contagens não substituem taxas por segmento.
for coluna in tabela.columns:
    if coluna != "cancelou":
        px.histogram(tabela, x=coluna, color="cancelou", barmode="group", width=700).show()


In [ ]:
# Mesmo recorte da execução histórica, sem sobrescrever a base completa.
recorte = tabela.loc[
    (tabela["duracao_contrato"] != "Monthly")
    & (tabela["ligacoes_callcenter"] < 5)
    & (tabela["dias_atraso"] <= 20)
].copy()
comparacao = pd.DataFrame([
    {"amostra": nome, "clientes": len(df), "cancelamentos": df["cancelou"].sum(), "taxa_pct": df["cancelou"].mean() * 100}
    for nome, df in [("Base completa após limpeza", tabela), ("Sem mensais", sem_mensais), ("Recorte combinado", recorte)]
])
display(comparacao)
print("Comparação descritiva entre populações diferentes; não mede impacto de retenção.")


## Limitações e próximos passos

Os limites de 5 ligações e 20 dias foram escolhidos explorando a própria base. Precisam de validação independente. Não há teste causal, modelo preditivo ou resultado empresarial. A origem, licença e disponibilização da base devem ser resolvidas antes de afirmar reprodução integral. Próximas análises: qualidade da base, taxas por segmento com denominadores e avaliação de confundidores.
